In [1]:
import os

In [2]:
%pwd

'/Users/dhruvjangid/Text-Summariser-/research'

In [3]:
os.chdir("../")

In [4]:
%pwd

'/Users/dhruvjangid/Text-Summariser-'

In [5]:
from dataclasses import dataclass 
from pathlib import Path


@dataclass(frozen=True)
class DataTransformationConfig:
    root_dir: Path
    data_path: Path
    tokenizer_name: Path

In [6]:
from textsummarizer.constants import *
from textsummarizer.utils.common import read_yaml, create_directories

In [8]:
class ConfigurationManager:
    def __init__(
        self,
        config_filepath = CONFIG_FILE_PATH,
        params_filepath = PARAMS_FILE_PATH):
        
        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)
        
        create_directories([self.config.artifacts_root])
        
    def get_data_transformation_config(self) -> DataTransformationConfig:
        config = self.config.data_transformation
        
        create_directories([config.root_dir])
        
        data_transformation_config = DataTransformationConfig(
            root_dir=config.root_dir,
            data_path=config.data_path,
            tokenizer_name=config.tokenizer_name
        )    
        
        return data_transformation_config    

In [9]:
import os 
from textsummarizer.logging import logger
from transformers import AutoTokenizer
from datasets import load_dataset, load_from_disk 


/Users/dhruvjangid/Text-Summariser-/textS/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


[2026-05-02 19:01:53,148: INFO: config: PyTorch version 2.11.0 available.]


In [15]:
class DataTransformation:
    def __init__(self, config):
        self.config = config
        self.tokenizer = AutoTokenizer.from_pretrained(config.tokenizer_name)

    def convert_examples_to_features(self, example_batch):
        model_inputs = self.tokenizer(
            ["summarize: " + d for d in example_batch["dialogue"]],
            max_length=512,
            truncation=True,
            padding="max_length"
        )

        labels = self.tokenizer(
            text_target=example_batch["summary"],
            max_length=64,
            truncation=True,
            padding="max_length"
        )

        labels_ids = [
            [(t if t != self.tokenizer.pad_token_id else -100) for t in label]
            for label in labels["input_ids"]
        ]

        model_inputs["labels"] = labels_ids
        return model_inputs

    def convert(self):  
        dataset_samsum = load_dataset("knkarthick/samsum")

        dataset_samsum_pt = dataset_samsum.map(
            self.convert_examples_to_features,
            batched=True
        )

        dataset_samsum_pt.save_to_disk(
            os.path.join(self.config.root_dir, "samsum_dataset")
        )

In [16]:
try: 
    config = ConfigurationManager()
    data_transformation_config = config.get_data_transformation_config()
    data_transformation = DataTransformation(config=data_transformation_config)
    data_transformation.convert()
except Exception as e:
    raise e

[2026-05-02 19:31:03,924: INFO: common: yaml file:config/config.yamlloaded successfully]
[2026-05-02 19:31:03,927: INFO: common: yaml file:params.yamlloaded successfully]
[2026-05-02 19:31:03,928: INFO: common: created directory at: artifacts]
[2026-05-02 19:31:03,929: INFO: common: created directory at: artifacts/data_transformation]


tokenizer_config.json: 2.32kB [00:00, 2.83MB/s]
spiece.model: 100%|██████████| 792k/792k [00:00<00:00, 3.16MB/s]
tokenizer.json: 1.39MB [00:00, 1.68MB/s]
Saving the dataset (1/1 shards): 100%|██████████| 819/819 [00:00<00:00, 205156.17 examples/s]
